In [7]:
import pandas as pd
import glob
import os

DATA_DIR = "../processed data"

event_files = glob.glob(os.path.join(DATA_DIR, "*Events_Cleaned.csv"))
tracking_files = glob.glob(os.path.join(DATA_DIR, "*Tracking_Cleaned.csv"))

print("Event files:", len(event_files))
print("Tracking files:", len(tracking_files))

events = pd.concat([pd.read_csv(f) for f in event_files], ignore_index=True)
tracking = pd.concat([pd.read_csv(f) for f in tracking_files], ignore_index=True)

print(events.shape)
print(tracking.shape)

Event files: 10
Tracking files: 10
(19200, 26)
(8063972, 14)


In [8]:
events.head()
tracking.head()

events.columns
tracking.columns

Index(['Image Id', 'Period', 'Game Clock', 'Player or Puck', 'Team',
       'Player Id', 'Player Jersey Number', 'Rink Location X (Feet)',
       'Rink Location Y (Feet)', 'Rink Location Z (Feet)', 'Player_Id_Cleaned',
       'Team_Name', 'Seconds', 'shift_flag'],
      dtype='str')

In [9]:
print(events.columns.tolist())
print(tracking.columns.tolist())

['Date', 'Home_Team', 'Away_Team', 'Period', 'Clock', 'Home_Team_Skaters', 'Away_Team_Skaters', 'Home_Team_Goals', 'Away_Team_Goals', 'Team', 'Player_Id', 'Event', 'X_Coordinate', 'Y_Coordinate', 'Detail_1', 'Detail_2', 'Detail_3', 'Detail_4', 'Player_Id_2', 'X_Coordinate_2', 'Y_Coordinate_2', 'Seconds', 'Strength', 'shift_flag', 'Next_Team', 'Next_Event']
['Image Id', 'Period', 'Game Clock', 'Player or Puck', 'Team', 'Player Id', 'Player Jersey Number', 'Rink Location X (Feet)', 'Rink Location Y (Feet)', 'Rink Location Z (Feet)', 'Player_Id_Cleaned', 'Team_Name', 'Seconds', 'shift_flag']


In [ ]:
#convert tracking columns to more standard names and ensure numeric types for analysis
trk = tracking.copy()

trk = trk.rename(columns={
    "Player_Id_Cleaned": "player_id",
    "Team_Name": "team",
    "Rink Location X (Feet)": "x",
    "Rink Location Y (Feet)": "y",
    "Rink Location Z (Feet)": "z",
    "Seconds": "seconds",
    "Player or Puck": "object_type",
    "Image Id": "frame_id"
})

trk["seconds"] = pd.to_numeric(trk["seconds"], errors="coerce")
trk["x"] = pd.to_numeric(trk["x"], errors="coerce")
trk["y"] = pd.to_numeric(trk["y"], errors="coerce")

trk = trk.dropna(subset=["seconds", "x", "y", "team"])

In [11]:
# separate player and puck tracking data 
players = trk[trk["object_type"].str.lower() != "puck"].copy()
puck = trk[trk["object_type"].str.lower() == "puck"].copy()

In [13]:
import numpy as np

In [15]:
# calculate player velocities and speed
players = players.sort_values(["player_id", "Period", "seconds"])

players["dt"] = players.groupby(["player_id", "Period"])["seconds"].diff()
players["dx"] = players.groupby(["player_id", "Period"])["x"].diff()
players["dy"] = players.groupby(["player_id", "Period"])["y"].diff()

players["vx"] = players["dx"] / players["dt"]
players["vy"] = players["dy"] / players["dt"]
players["speed"] = np.sqrt(players["vx"]**2 + players["vy"]**2)

# remove any rows with infinite or NaN speeds
players.loc[(players["dt"] <= 0) | (players["dt"] > 2), ["vx", "vy", "speed"]] = np.nan

Feature Engineering: A rush should be one row in a final rush-level table.


In [16]:
# support distance: to identify when players are close to the puck and could potentially receive a pass

def nearest_teammate_distance(df_team, carrier_id):
    carrier = df_team[df_team["player_id"] == carrier_id][["x", "y"]].iloc[0]
    mates = df_team[df_team["player_id"] != carrier_id].copy()
    if mates.empty:
        return np.nan
    d = np.sqrt((mates["x"] - carrier["x"])**2 + (mates["y"] - carrier["y"])**2)
    return d.min()

In [ ]:
# lane structure: use Y positions of the offensive team (5 playsers)

def lane_features(df_team):
    y = df_team["y"].dropna().values
    if len(y) < 3:
        return {"y_std": np.nan, "y_range": np.nan}
    return {
        "y_std": np.std(y),
        "y_range": np.max(y) - np.min(y)
    }

In [18]:
# depth layering: use X positions to see if players are spread out in depth or clustered together

def depth_features(df_team):
    x = np.sort(df_team["x"].dropna().values)
    if len(x) < 3:
        return {"x_std": np.nan, "depth_range": np.nan, "flat_line": np.nan}
    diffs = np.diff(x)
    return {
        "x_std": np.std(x),
        "depth_range": np.max(x) - np.min(x),
        "flat_line": int((np.max(x) - np.min(x)) < 10)
    }

In [ ]:
# speed and variability: average speed and how much it varies, as well as variability in X velocity to see if players are changing their depth position
def speed_features(df_team):
    vx = df_team["vx"].dropna().values
    speed = df_team["speed"].dropna().values
    if len(speed) < 2:
        return {"mean_speed": np.nan, "speed_var": np.nan, "vx_var": np.nan}
    return {
        "mean_speed": np.mean(speed),
        "speed_var": np.var(speed),
        "vx_var": np.var(vx)
    }

In [20]:
feature_cols = [
    "nearest_support_dist",
    "n_teammates_within_20",
    "y_std",
    "depth_range",
    "flat_line",
    "mean_speed",
    "speed_var"
]